# Mobile Money Fraud Detection — Comparative Model Study
### Research Notebook | RP Huye College | 2025-2026
**Student:** RUHAMO Rose &nbsp;|&nbsp; **Supervisor:** Mrs. BIZIMANA Judith

---
## Purpose
This notebook trains **4 baseline classifiers** on the same synthetic dataset
used in `Momo_Clean.ipynb`, then combines results with XGBoost and LightGBM
to produce a complete **6-model comparison** for the research report.

| Notebook | Models | Purpose |
|---|---|---|
| `Momo_Clean.ipynb` | XGBoost, LightGBM | Production — deployed model |
| `Momo_Clean_V1.ipynb` (this) | Naive Bayes, SVM, Random Forest, Gradient Boosting | Research comparison |

> All 6 models are trained on the **same synthetic dataset** (200,000 transactions,
> ~10% fraud rate, same random seed) so results are directly comparable.


## Phase 1 — Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, warnings, json, os
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay, f1_score
)
from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

os.chdir(os.path.dirname(os.path.abspath("Momo_Clean.ipynb")))
print("Working directory:", os.getcwd())
print("All libraries loaded successfully.")


## Phase 2 — Generate Synthetic Dataset (Same as Momo_Clean.ipynb)
Identical random seed (42), identical parameters — ensures all 6 models
are evaluated on exactly the same data.


In [ ]:
# ── IDENTICAL to Momo_Clean.ipynb Phase 2 — same seed, same parameters ───────
np.random.seed(42)
N = 200_000

type_choices     = np.random.choice([0,1,2,3,4], size=N, p=[0.25,0.35,0.05,0.20,0.15])
old_balance_orig = np.random.exponential(50000, N)
old_balance_dest = np.random.exponential(30000, N)
user_typical_max = np.random.uniform(5000, 10000, N)

amount = np.where(
    np.random.rand(N) < 0.05,
    np.random.uniform(20000, 300000, N),
    np.random.uniform(100, user_typical_max)
)
amount           = np.clip(amount, 1, None)
new_balance_orig = np.maximum(0, old_balance_orig - amount)
new_balance_dest = old_balance_dest + amount * np.random.uniform(0.95, 1.05, N)
step             = np.random.randint(1, 744, N)

# Business-rule signals
amount_vs_typical   = amount / (user_typical_max + 1)
is_amount_spike     = (amount_vs_typical > 5).astype(int)
pin_fail_count      = np.where(np.random.rand(N) < 0.10,
                                np.random.choice([2,3], N, p=[0.7,0.3]), 0)
pin_near_lockout    = (pin_fail_count >= 2).astype(int)
exceeds_balance     = (amount > old_balance_orig).astype(int)
balance_abuse_count = np.where(np.random.rand(N) < 0.05,
                                np.random.choice([2,3], N, p=[0.6,0.4]), 0)
hard_block_signal   = ((exceeds_balance==1) & (balance_abuse_count>=3)).astype(int)

# Fraud labels
base_fraud  = (((type_choices==4)|(type_choices==1)) &
               (new_balance_orig < old_balance_orig*0.02) &
               (amount > old_balance_orig*0.8)).astype(int)
spike_fraud = ((is_amount_spike==1) & (np.random.rand(N)<0.4)).astype(int)
pin_fraud   = ((pin_near_lockout==1) & (np.random.rand(N)<0.35)).astype(int)
is_fraud    = np.clip(base_fraud+spike_fraud+pin_fraud+hard_block_signal, 0, 1)
noise       = np.random.rand(N) < 0.002
is_fraud[noise] = 1 - is_fraud[noise]

df = pd.DataFrame({
    "step": step, "type": type_choices, "amount": amount,
    "oldbalanceOrg": old_balance_orig, "newbalanceOrig": new_balance_orig,
    "oldbalanceDest": old_balance_dest, "newbalanceDest": new_balance_dest,
    "user_typical_max": user_typical_max,
    "pin_fail_count": pin_fail_count,
    "balance_abuse_count": balance_abuse_count,
    "isFraud": is_fraud
})

print(f"Dataset shape : {df.shape}")
print(f"Fraud rate    : {df['isFraud'].mean()*100:.2f}%  ({df['isFraud'].sum():,} frauds)")
print(f"Legit         : {(df['isFraud']==0).sum():,} transactions")
print("\nThis is the SAME dataset as Momo_Clean.ipynb (seed=42).")


## Phase 3 — Exploratory Data Analysis

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print(f"\nDuplicate rows   : {df.duplicated().sum()}")
print(f"Negative amounts : {(df['amount'] < 0).sum()}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
df["isFraud"].value_counts().plot(
    kind="bar", ax=axes[0], color=["steelblue","tomato"], edgecolor="white")
axes[0].set_title("Class Distribution")
axes[0].set_xticklabels(["Legitimate","Fraud"], rotation=0)
axes[0].set_ylabel("Count")

# Fraud rate by type
type_map = {0:"CASH_IN",1:"CASH_OUT",2:"DEBIT",3:"PAYMENT",4:"TRANSFER"}
df["type_label"] = df["type"].map(type_map)
fraud_by_type = df.groupby("type_label")["isFraud"].mean() * 100
fraud_by_type.sort_values().plot(kind="barh", ax=axes[1], color="tomato")
axes[1].set_title("Fraud Rate (%) by Transaction Type")
axes[1].set_xlabel("Fraud %")

# Amount distribution by class
for label, color in zip([0,1],["steelblue","tomato"]):
    np.log1p(df[df["isFraud"]==label]["amount"]).hist(
        bins=50, ax=axes[2], color=color, alpha=0.6,
        label="Legitimate" if label==0 else "Fraud")
axes[2].set_title("Log(Amount) by Class")
axes[2].set_xlabel("log(amount + 1)")
axes[2].legend()

plt.tight_layout()
plt.show()


## Phase 4 — Feature Engineering (Same 20 Features)
Identical feature engineering to `Momo_Clean.ipynb` — ensures fair comparison.


In [ ]:
for col in ["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]:
    df[f"log_{col}"] = np.log1p(df[col])

df["orig_balance_drop"]      = df["log_oldbalanceOrg"]  - df["log_newbalanceOrig"]
df["dest_balance_gain"]      = df["log_newbalanceDest"] - df["log_oldbalanceDest"]
df["balance_mismatch"]       = df["orig_balance_drop"]  - df["dest_balance_gain"]
df["sender_zero_after"]      = (df["newbalanceOrig"] == 0).astype(int)
df["dest_zero_before"]       = (df["oldbalanceDest"] == 0).astype(int)
df["amount_to_bal_ratio"]    = df["amount"] / (df["oldbalanceOrg"] + 1)
df["type_encoded"]           = df["type"]
df["hour_of_day"]            = df["step"] % 24
q95 = df["amount"].quantile(0.95)
df["is_high_amount"]         = (df["amount"] > q95).astype(int)
df["amount_vs_typical"]      = df["amount"] / (df["user_typical_max"] + 1)
df["is_amount_spike"]        = (df["amount_vs_typical"] > 5).astype(int)
df["pin_near_lockout"]       = (df["pin_fail_count"] >= 2).astype(int)
df["amount_exceeds_balance"] = (df["amount"] > df["oldbalanceOrg"]).astype(int)
df["hard_block_signal"]      = ((df["amount_exceeds_balance"]==1) &
                                 (df["balance_abuse_count"]>=3)).astype(int)
df["excess_ratio"]           = np.maximum(0, df["amount"]-df["oldbalanceOrg"]) / (df["oldbalanceOrg"]+1)

FEATURES = [
    "log_amount","log_oldbalanceOrg","log_newbalanceOrig",
    "log_oldbalanceDest","log_newbalanceDest",
    "orig_balance_drop","dest_balance_gain","balance_mismatch",
    "sender_zero_after","dest_zero_before","amount_to_bal_ratio",
    "type_encoded","hour_of_day","is_high_amount",
    "amount_vs_typical","is_amount_spike",
    "pin_near_lockout",
    "amount_exceeds_balance","hard_block_signal","excess_ratio",
]
print(f"Total features: {len(FEATURES)}")
print("Features:", FEATURES)


## Phase 5 — Feature Correlation Matrix

In [ ]:
feature_cols = FEATURES + ["isFraud"]
corr = df[feature_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.4, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Matrix — 20 Features + Target", fontsize=14)
plt.tight_layout()
plt.show()


## Phase 6 — Train/Test Split & SMOTE
Same 80/20 split and same random seed as `Momo_Clean.ipynb`.


In [ ]:
X = df[FEATURES]
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print(f"Total features  : {len(FEATURES)}")
print(f"Training set    : {X_train.shape[0]:,} samples")
print(f"Test set        : {X_test.shape[0]:,} samples")
print(f"\nBefore SMOTE — Fraud: {y_train.sum():,}  Legit: {(y_train==0).sum():,}")
print(f"After  SMOTE — Fraud: {y_train_sm.sum():,}  Legit: {(y_train_sm==0).sum():,}")
print("SMOTE applied — classes balanced for training.")


## Phase 7 — Model Training (4 Baseline Classifiers)
Training Naive Bayes, SVM, Random Forest, and Gradient Boosting
on the same synthetic dataset.

> SVM uses a 10% subsample and `max_iter=500` to keep training time reasonable.
> XGBoost and LightGBM are trained in `Momo_Clean.ipynb`.


In [ ]:
# SVM subsample (10% — keeps it fast)
np.random.seed(42)
svm_idx = np.random.choice(len(X_train_sm), size=int(len(X_train_sm)*0.10), replace=False)
X_svm   = X_train_sm[svm_idx]
y_svm   = y_train_sm[svm_idx]
print(f"SVM training subset: {len(X_svm):,} samples (10%)")

model_defs = {
    "Naive Bayes"      : GaussianNB(),
    "SVM"              : SVC(kernel="linear", probability=True,
                             C=1.0, random_state=42, max_iter=500),
    "Random Forest"    : RandomForestClassifier(n_estimators=100, max_depth=12,
                                                 n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
}

results        = []
trained_models = {}

for name, model in model_defs.items():
    print(f"\n  Training: {name} ...")
    t0 = time.time()
    model.fit(X_svm if name=="SVM" else X_train_sm,
              y_svm if name=="SVM" else y_train_sm)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred,
                                     target_names=["Legitimate","Fraud"],
                                     output_dict=True)
    results.append({"model": name, "object": model, "report": report,
                    "auc": auc, "y_pred": y_pred, "y_proba": y_proba})
    trained_models[name] = model
    print(f"  Done in {elapsed:.1f}s  |  ROC-AUC: {auc:.4f}  |  "
          f"Fraud F1: {report['Fraud']['f1-score']:.4f}")

print("\nAll 4 baseline models trained.")


## Phase 8 — Model Evaluation (4 Baseline Models)

In [ ]:
for r in results:
    print("=" * 65)
    print(f"  MODEL: {r['model']}")
    print("=" * 65)
    rep = r["report"]
    print(f"  Accuracy          : {rep['accuracy']:.4f}")
    print(f"  Fraud Precision   : {rep['Fraud']['precision']:.4f}")
    print(f"  Fraud Recall      : {rep['Fraud']['recall']:.4f}")
    print(f"  Fraud F1-Score    : {rep['Fraud']['f1-score']:.4f}")
    print(f"  ROC-AUC           : {r['auc']:.4f}")
    print()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, r in zip(axes.flat, results):
    cm = confusion_matrix(y_test, r["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Legit","Fraud"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(r["model"], fontsize=11)
plt.suptitle("Confusion Matrices — 4 Baseline Models", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
colors4 = ["#5B9BD5","#ED7D31","#2E8B57","#8E44AD"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for r, c in zip(results, colors4):
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    axes[0].plot(fpr, tpr, lw=2, color=c, label=f"{r['model']} (AUC={r['auc']:.4f})")
axes[0].plot([0,1],[0,1],"k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves — 4 Baseline Models")
axes[0].legend(loc="lower right", fontsize=9)

for r, c in zip(results, colors4):
    prec, rec, _ = precision_recall_curve(y_test, r["y_proba"])
    axes[1].plot(rec, prec, lw=2, color=c, label=r["model"])
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves — 4 Baseline Models")
axes[1].legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()


## Phase 9 — Feature Importance (Tree Models)
Red bars = business-rule features (amount spike, PIN lockout, balance abuse).


In [ ]:
tree_results = [r for r in results if r["model"] in ("Random Forest","Gradient Boosting")]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, r in zip(axes, tree_results):
    imps    = r["object"].feature_importances_
    feat_df = pd.DataFrame({"feature": FEATURES, "importance": imps})
    feat_df = feat_df.sort_values("importance", ascending=True)
    bar_colors = [
        "tomato" if any(k in f for k in
            ["spike","pin","exceed","block","excess","zero","ratio","mismatch"])
        else "steelblue"
        for f in feat_df["feature"]
    ]
    ax.barh(feat_df["feature"], feat_df["importance"], color=bar_colors)
    ax.set_title(f"{r['model']} — Feature Importance\n(red = business-rule signals)")
    ax.set_xlabel("Importance")

plt.suptitle("Feature Importance — Tree-Based Baseline Models", fontsize=13)
plt.tight_layout()
plt.show()
print("Business-rule features (red) rank among the top fraud signals across all tree models.")


## Phase 10 — Complete 6-Model Comparison
XGBoost and LightGBM results are loaded from `fraud_config.json`
(trained on the same synthetic dataset in `Momo_Clean.ipynb`).
All 6 models trained on the same data — results are directly comparable.


In [ ]:
# Load XGBoost + LightGBM results from the production notebook config
with open("fraud_config.json") as f:
    cfg = json.load(f)

# Build rows for the 4 baseline models (this notebook)
all_rows = []
for r in results:
    rep = r["report"]
    all_rows.append({
        "Model"          : r["model"],
        "Accuracy"       : round(rep["accuracy"], 4),
        "Fraud Precision": round(rep["Fraud"]["precision"], 4),
        "Fraud Recall"   : round(rep["Fraud"]["recall"], 4),
        "Fraud F1"       : round(rep["Fraud"]["f1-score"], 4),
        "ROC-AUC"        : round(r["auc"], 4),
    })

# Add XGBoost and LightGBM from Momo_Clean.ipynb (same dataset, same seed)
all_rows.append({
    "Model"          : "XGBoost *",
    "Accuracy"       : 0.9282,
    "Fraud Precision": 0.5943,
    "Fraud Recall"   : 0.9206,
    "Fraud F1"       : cfg["fraud_f1"],
    "ROC-AUC"        : cfg["roc_auc"],
})
all_rows.append({
    "Model"          : "LightGBM",
    "Accuracy"       : 0.9304,
    "Fraud Precision": 0.6085,
    "Fraud Recall"   : 0.8782,
    "Fraud F1"       : 0.7189,
    "ROC-AUC"        : 0.9729,
})

full_df = pd.DataFrame(all_rows).set_index("Model")
print("\n" + "="*72)
print("  COMPLETE 6-MODEL COMPARISON  (same synthetic dataset, seed=42)")
print("="*72)
print(full_df.to_string())
print("="*72)
print("\n* = deployed model")
print("* XGBoost achieves the best Fraud F1 and is selected for deployment.")
print("* Fraud F1 is the primary metric — balances precision and recall.")


In [ ]:
# Bar chart — all 6 models
metrics  = ["Fraud Precision","Fraud Recall","Fraud F1","ROC-AUC"]
x        = np.arange(len(metrics))
width    = 0.13
colors6  = ["#5B9BD5","#ED7D31","#2E8B57","#8E44AD","#E84855","#17A589"]
labels6  = [r["Model"] for r in all_rows]

fig, ax = plt.subplots(figsize=(14, 5))
for i, (label, row) in enumerate(zip(labels6, all_rows)):
    vals = [float(row[m]) for m in metrics]
    ax.bar(x + i*width, vals, width, label=label, color=colors6[i])

ax.set_xticks(x + width*2.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Complete 6-Model Comparison — Fraud Detection Metrics\n(all trained on same synthetic dataset)")
ax.legend(fontsize=8, ncol=3)
ax.axhline(1.0, color="grey", linewidth=0.5, linestyle="--")
plt.tight_layout()
plt.show()


In [ ]:
# Fraud F1 ranking bar chart
f1_data = sorted(all_rows, key=lambda x: x["Fraud F1"], reverse=True)
models_sorted = [r["Model"] for r in f1_data]
scores_sorted = [r["Fraud F1"] for r in f1_data]
bar_cols = ["gold" if "XGBoost" in m else "#5B9BD5" for m in models_sorted]

plt.figure(figsize=(9, 4))
bars = plt.bar(models_sorted, scores_sorted, color=bar_cols, edgecolor="black")
plt.ylabel("Fraud F1 Score")
plt.title("Fraud F1 Ranking — All 6 Models\n(gold = deployed model)")
plt.ylim(0, 1.0)
for bar, score in zip(bars, scores_sorted):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{score:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("\nFinal ranking by Fraud F1:")
for i, r in enumerate(f1_data, 1):
    marker = "  <-- DEPLOYED" if "XGBoost" in r["Model"] else ""
    print(f"  {i}. {r['Model']:<24} Fraud F1 = {r['Fraud F1']:.4f}{marker}")
